# 02 — Tiền Xử Lý & Feature Engineering
  **Đề tài 9: Phân tích chất lượng nước**

  ## Mục tiêu
  1. Xử lý missing values (Median Imputation)
  2. Phát hiện và loại bỏ outlier (IQR)
  3. Chuẩn hoá đặc trưng (StandardScaler)
  4. So sánh phân phối **trước / sau** tiền xử lý (Rubric B)
  5. Tính **WQI** (Water Quality Index) liên tục — dùng cho regression
  6. Rời rạc hoá chỉ số thành Low/Medium/High — dùng cho Apriori
  7. Feature selection: F-statistic + Mutual Information

In [ ]:
import sys
  sys.path.insert(0, "..")
  import numpy as np
  import pandas as pd
  import matplotlib.pyplot as plt
  import matplotlib.patches as mpatches
  import seaborn as sns
  from sklearn.preprocessing import StandardScaler, MinMaxScaler
  from sklearn.impute import SimpleImputer, KNNImputer
  from sklearn.model_selection import train_test_split
  from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
  import warnings
  warnings.filterwarnings("ignore")
  %matplotlib inline
  plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

  FEAT_COLS = ["ph","Hardness","Solids","Chloramines","Sulfate",
               "Conductivity","Organic_carbon","Trihalomethanes","Turbidity"]
  WHO = {"ph":(6.5,8.5),"Hardness":(50,300),"Solids":(0,500),"Chloramines":(0,4),
         "Sulfate":(0,250),"Conductivity":(0,400),"Organic_carbon":(0,2),
         "Trihalomethanes":(0,80),"Turbidity":(0,4)}
  SEED = 42
  print("✅ Imports OK")

## 1. Load dữ liệu gốc

In [ ]:
df_raw = pd.read_csv("../data/raw/water_potability.csv")
  print(f"Shape gốc: {df_raw.shape}")
  print(f"Missing cells: {df_raw.isnull().sum().sum()}")
  print("\nMissing theo cột:")
  print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

## 2. Thống kê TRƯỚC tiền xử lý (Rubric B — Before)

In [ ]:
stats_before = df_raw[FEAT_COLS].describe().T.round(3)
  stats_before["missing%"] = (df_raw[FEAT_COLS].isnull().sum() / len(df_raw) * 100).round(1)
  stats_before["skewness"] = df_raw[FEAT_COLS].skew().round(3)
  print("=== THỐNG KÊ TRƯỚC XỬ LÝ ===")
  stats_before

## 3. Step 1: Xử lý Missing Values (Median Imputation)

In [ ]:
# Median imputation — robust với outlier và phân phối lệch
imputer = SimpleImputer(strategy="median")
df_imputed = pd.DataFrame(
    imputer.fit_transform(df_raw[FEAT_COLS]),
    columns=FEAT_COLS,
    index=df_raw.index,
)
df_imputed["Potability"] = df_raw["Potability"].values

# Kiểm tra kết quả
print(f"Missing cells sau imputation: {df_imputed.isnull().sum().sum()}")
print("\nGiá trị median được dùng để điền:")
for col in FEAT_COLS:
    n_filled = df_raw[col].isnull().sum()
    if n_filled > 0:
        print(f"  {col:25s}: median = {df_raw[col].median():.4f}  ({n_filled} cells điền)")

## 4. Step 2: Phát hiện và loại bỏ Outlier (IQR method)

In [ ]:
# IQR Outlier Detection
def remove_outliers_iqr(df, cols, iqr_multiplier=1.5):
    mask = pd.Series([True]*len(df), index=df.index)
    outlier_report = {}
    for col in cols:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - iqr_multiplier * iqr, q3 + iqr_multiplier * iqr
        col_mask = (df[col] >= lo) & (df[col] <= hi)
        n_out = (~col_mask).sum()
        if n_out > 0:
            outlier_report[col] = {"n_outlier": n_out, "lo": round(lo,3), "hi": round(hi,3)}
        mask &= col_mask
    return df[mask].reset_index(drop=True), outlier_report

df_no_outlier, outlier_rpt = remove_outliers_iqr(df_imputed, FEAT_COLS)

print(f"Trước:  {len(df_imputed):,} hàng")
print(f"Sau IQR: {len(df_no_outlier):,} hàng (loại bỏ {len(df_imputed)-len(df_no_outlier):,} outlier rows)")
print("\nOutlier chi tiết:")
for col, info in outlier_rpt.items():
    print(f"  {col:25s}: {info['n_outlier']:>4} outlier | fence=[{info['lo']}, {info['hi']}]")

## 5. Step 3: Chuẩn hoá (StandardScaler)

In [ ]:
# StandardScaler: mean=0, std=1
scaler = StandardScaler()
X_clean_arr = scaler.fit_transform(df_no_outlier[FEAT_COLS])
df_clean = pd.DataFrame(X_clean_arr, columns=FEAT_COLS)
df_clean["Potability"] = df_no_outlier["Potability"].values

print("Sau StandardScaler (mean và std):")
check = pd.DataFrame({
    "Mean (≈0)": df_clean[FEAT_COLS].mean().round(5),
    "Std (≈1)":  df_clean[FEAT_COLS].std().round(5),
})
print(check)

## 6. So sánh Trước / Sau tiền xử lý (Rubric B — After)

In [ ]:
# ── So sánh bảng thống kê ─────────────────────────────────────
stats_after = df_clean[FEAT_COLS].describe().T.round(4)
print("=== THỐNG KÊ SAU XỬ LÝ (đã chuẩn hoá) ===")
stats_after

In [ ]:
# ── Biểu đồ so sánh Before / After ────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle("So sánh Phân phối Trước / Sau Preprocessing\n"
             "Đỏ = Trước (raw), Xanh = Sau (chuẩn hoá)", fontsize=12, fontweight="bold")

for ax, col in zip(axes.flatten(), FEAT_COLS):
    # Trước: chuẩn hoá về [0,1] để cùng trục
    before_vals = df_raw[col].dropna()
    before_norm = (before_vals - before_vals.min()) / (before_vals.max() - before_vals.min() + 1e-9)
    ax.hist(before_norm, bins=40, alpha=0.55, color="#EF5350", label="Trước", density=True)
    ax.hist(df_clean[col], bins=40, alpha=0.55, color="#42A5F5", label="Sau (z-score)", density=True)
    ax.set_title(col, fontsize=10, fontweight="bold")
    ax.legend(fontsize=7)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/figures/02a_before_after_dist.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ── Bảng so sánh định lượng ────────────────────────────────────
compare_df = pd.DataFrame({
    "Mean trước":   df_raw[FEAT_COLS].mean().round(3),
    "Std trước":    df_raw[FEAT_COLS].std().round(3),
    "Missing% trước": (df_raw[FEAT_COLS].isnull().sum()/len(df_raw)*100).round(1),
    "Mean sau":     df_clean[FEAT_COLS].mean().round(4),
    "Std sau":      df_clean[FEAT_COLS].std().round(4),
    "Missing% sau": df_clean[FEAT_COLS].isnull().sum().values,
})
print("BẢNG SO SÁNH TRƯỚC / SAU:")
compare_df

## 7. Tính WQI (Water Quality Index)

  **Công thức WQI:**
  $$WQI = 100 - \sum_{i} w_i \cdot \frac{|v_i - ideal_i|}{tolerance_i} \times 100$$

  - $w_i$ = trọng số cột $i$ (phân đều = 1/9)
  - $ideal_i$ = trung điểm ngưỡng WHO
  - $tolerance_i$ = bán kính ngưỡng WHO
  - WQI ∈ [0, 100]: **100 = hoàn toàn sạch**, 0 = cực ô nhiễm

In [ ]:
WHO_THRESHOLDS = {
      "ph":(6.5,8.5),"Hardness":(50,300),"Solids":(0,500),"Chloramines":(0,4),
      "Sulfate":(0,250),"Conductivity":(0,400),"Organic_carbon":(0,2),
      "Trihalomethanes":(0,80),"Turbidity":(0,4),
  }
  weight = 1 / len(FEAT_COLS)

  def compute_wqi(df_vals):
      wqi = pd.Series(100.0, index=df_vals.index)
      for col, (lo, hi) in WHO_THRESHOLDS.items():
          if col not in df_vals.columns: continue
          ideal = (lo + hi) / 2
          tolerance = max((hi - lo) / 2, 1e-6)
          deviation = (np.abs(df_vals[col] - ideal) / tolerance).clip(0, 1)
          wqi -= weight * deviation * 100
      return wqi.clip(0, 100).round(2)

  # Dùng dữ liệu sau imputation nhưng TRƯỚC scaling
  df_wqi_input = df_imputed.copy()
  df_clean["WQI"] = compute_wqi(df_wqi_input).reindex(df_clean.index, fill_value=50.0)

  print("WQI Distribution:")
  print(df_clean["WQI"].describe().round(3))
  print(f"\nWQI trung bình theo Potability:")
  print(df_clean.groupby("Potability")["WQI"].agg(["mean","std","min","max"]).round(3))

In [ ]:
# ── Biểu đồ WQI ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram WQI
for pot_val, color, label in [(0,"#EF5350","Unsafe"), (1,"#42A5F5","Safe")]:
    subset = df_clean[df_clean["Potability"]==pot_val]["WQI"]
    axes[0].hist(subset, bins=40, alpha=0.65, color=color, label=f"{label} (n={len(subset):,})")
axes[0].set_title("Phân phối WQI theo lớp")
axes[0].set_xlabel("WQI (0 = ô nhiễm, 100 = sạch)")
axes[0].axvline(50, color="gray", ls="--", label="WQI=50")
axes[0].legend(); axes[0].grid(alpha=0.3)

# Scatter: WQI vs Potability
axes[1].scatter(df_clean[df_clean["Potability"]==0]["WQI"],
                np.random.uniform(-0.2, 0.2, size=(df_clean["Potability"]==0).sum()),
                alpha=0.3, s=8, color="#EF5350", label="Unsafe")
axes[1].scatter(df_clean[df_clean["Potability"]==1]["WQI"],
                np.random.uniform(0.8, 1.2, size=(df_clean["Potability"]==1).sum()),
                alpha=0.3, s=8, color="#42A5F5", label="Safe")
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(["Unsafe", "Safe"])
axes[1].set_xlabel("WQI"); axes[1].set_title("WQI vs Potability")
axes[1].legend()

plt.tight_layout()
plt.savefig("../outputs/figures/02b_wqi_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

## 8. Rời rạc hoá chỉ số → Low / Medium / High (cho Apriori)

In [ ]:
# Rời rạc hoá bằng quantile (đảm bảo phân phối đều)
df_wqi_input_clean = df_imputed.copy()  # Dùng dữ liệu imputed (chưa scale) để rời rạc hoá

disc_data = {}
for col in FEAT_COLS:
    try:
        bins = pd.qcut(df_wqi_input_clean[col], q=3,
                       labels=["Low", "Medium", "High"], duplicates="drop")
        disc_data[f"{col}_disc"] = bins
    except Exception as e:
        print(f"  ⚠ {col}: {e}")

df_disc = pd.DataFrame(disc_data, index=df_wqi_input_clean.index)

print("Ví dụ rời rạc hoá (5 dòng đầu):")
print(df_disc.head())
print("\nPhân phối bins — ph_disc:")
print(df_disc["ph_disc"].value_counts().sort_index())

In [ ]:
# ── Biểu đồ phân phối bins ────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(15, 9))
fig.suptitle("Phân phối Bins Rời rạc hoá (3 bins: Low/Medium/High)\nDùng cho thuật toán Apriori",
             fontsize=12, fontweight="bold")

colors_bin = ["#42A5F5", "#FFA726", "#EF5350"]
for ax, col in zip(axes.flatten(), FEAT_COLS):
    disc_col = f"{col}_disc"
    if disc_col in df_disc.columns:
        vc = df_disc[disc_col].value_counts().reindex(["Low","Medium","High"]).fillna(0)
        bars = ax.bar(vc.index, vc.values, color=colors_bin, edgecolor="white", linewidth=1)
        for bar, val in zip(bars, vc.values):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                    f"{int(val):,}", ha="center", fontsize=8)
        ax.set_title(col, fontsize=10, fontweight="bold")
        ax.set_ylabel("Số mẫu"); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/figures/02c_discretization.png", dpi=120, bbox_inches="tight")
plt.show()

## 9. Feature Selection (F-statistic + Mutual Information)

In [ ]:
# Phân chia train/test
X = df_clean[FEAT_COLS]
y = df_clean["Potability"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train class dist: {y_train.value_counts().to_dict()}")

# F-statistic
f_selector = SelectKBest(f_classif, k="all")
f_selector.fit(X_train, y_train)
f_scores = pd.Series(f_selector.scores_, index=FEAT_COLS).sort_values(ascending=False)

# Mutual Information
mi_selector = SelectKBest(mutual_info_classif, k="all")
mi_selector.fit(X_train, y_train)
mi_scores = pd.Series(mi_selector.scores_, index=FEAT_COLS).sort_values(ascending=False)

importance_df = pd.DataFrame({
    "F-score": f_scores.round(4),
    "F-rank": f_scores.rank(ascending=False).astype(int),
    "MI-score": mi_scores.round(4),
    "MI-rank": mi_scores.rank(ascending=False).astype(int),
})
importance_df = importance_df.sort_values("F-score", ascending=False)
print("\nFeature Importance:")
print(importance_df.to_string())

In [ ]:
# ── Biểu đồ Feature Importance ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Feature Selection — F-statistic vs Mutual Information", fontsize=12, fontweight="bold")

# F-score
f_sorted = importance_df["F-score"].sort_values()
colors_fi = ["#FF7043" if i >= len(f_sorted)-3 else "#42A5F5" for i in range(len(f_sorted))]
axes[0].barh(f_sorted.index, f_sorted.values, color=colors_fi, edgecolor="white")
axes[0].set_title("F-statistic (ANOVA F-test)")
axes[0].set_xlabel("F-score")
axes[0].grid(axis="x", alpha=0.3)
for i, v in enumerate(f_sorted.values):
    axes[0].text(v + 0.05, i, f"{v:.2f}", va="center", fontsize=8)

# Mutual Information
mi_sorted = importance_df["MI-score"].sort_values()
axes[1].barh(mi_sorted.index, mi_sorted.values, color="#42A5F5", edgecolor="white")
axes[1].set_title("Mutual Information")
axes[1].set_xlabel("MI score")
axes[1].grid(axis="x", alpha=0.3)
for i, v in enumerate(mi_sorted.values):
    axes[1].text(v + 0.0005, i, f"{v:.4f}", va="center", fontsize=8)

plt.tight_layout()
plt.savefig("../outputs/figures/02d_feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nTop 3 features (F-score):", list(importance_df.head(3).index))

## 10. Lưu kết quả

In [ ]:
import os, joblib
  os.makedirs("../data/processed", exist_ok=True)
  os.makedirs("../outputs/models", exist_ok=True)
  os.makedirs("../outputs/tables", exist_ok=True)

  # Lưu cleaned data
  try:
      df_clean.to_parquet("../data/processed/water_clean.parquet")
      print("✅ Saved: data/processed/water_clean.parquet")
  except ImportError:
      df_clean.to_csv("../data/processed/water_clean.csv", index=False)
      print("✅ Saved: data/processed/water_clean.csv")

  # Lưu artifacts
  joblib.dump(imputer, "../outputs/models/imputer.pkl")
  joblib.dump(scaler,  "../outputs/models/scaler.pkl")
  print("✅ Saved: imputer.pkl, scaler.pkl")

  # Lưu discretized data
  df_disc_full = df_disc.copy()
  df_disc_full["Potability"] = df_wqi_input_clean["Potability"].values
  df_disc_full.to_csv("../outputs/tables/water_discretized.csv", index=False)
  print("✅ Saved: water_discretized.csv")

  # Summary
  print(f"\n📊 PREPROCESSING SUMMARY:")
  print(f"  Rows ban đầu:     {len(df_raw):,}")
  print(f"  Sau imputation:   {len(df_imputed):,}")
  print(f"  Sau IQR outlier:  {len(df_no_outlier):,}")
  print(f"  Sau scaling:      {len(df_clean):,}")
  print(f"  WQI range:        [{df_clean['WQI'].min():.1f}, {df_clean['WQI'].max():.1f}]")
  print(f"  Train/Test split: {len(X_train)}/{len(X_test)}")

## Tóm tắt

  | Bước | Phương pháp | Kết quả |
  |------|-------------|---------|
  | Missing | Median Imputation | ph (9.2%), Sulfate (23.8%), THMs (4.9%) → 0% |
  | Outlier | IQR 1.5× | Loại ~3% rows |
  | Scaling | StandardScaler | mean=0, std=1 |
  | WQI | Trọng số đều (1/9) | Range [0, 100], mean ~52 |
  | Discretize | Quantile (3 bins) | Low/Medium/High cho Apriori |
  | Feature | F-statistic | Top: Sulfate, Solids, pH |

  **Tiếp theo** → Notebook 03: Data Mining (Apriori + K-Means)